# Agilent BioTek MultiFlo dispenser quickstart

The MultiFlo is a bulk dispenser and nothing else: it fills wells from syringes, which meter a
volume precisely, and from peristaltic pumps, which push fluid through a tubing cassette. There is
no wash manifold, so washing a plate is not something this model does.

It is also the oldest firmware in the family, and that shows: it predates dispensing into
individually chosen wells and cannot store the fields for it, so such a step is refused rather than
run across the whole plate.

This quickstart connects to the dispenser, reads what it is and what it has fitted, tells it which
plate is on its carrier, primes and dispenses from both kinds of pump, explains where in the well a
step works, runs a protocol file, and disconnects.

| Property | Value |
|---|---|
| Communication | A serial port, or USB through an FTDI interface |
| Serial parameters | 38400 baud, 8 data bits, 2 stop bits, no parity, no flow control |
| Operations | Syringe priming and dispensing, peristaltic priming, purging and dispensing, shake and soak |
| Plate formats | 96- and 384-well including deep-well, 96 half-well, mini tubes, 1536-well |
| Syringes | A and B, each with two selectable bottles; both at once on the manifolds that allow it |
| Peristaltic pumps | Primary and secondary, one tubing cassette each |
| Dispenser reach | Depth 1-1500 motor steps; across the well ±125, along the well ±40 |
| Protocol files | `.LHC` protocol files are read, checked and run |

```{warning}
This driver has not yet been checked against a real MultiFlo. `setup()` says so in the log every
time it runs. Every step it sends has been checked against the vendor's own interface library —
frame for frame, and against the same validation the instrument applies — but that is not the same
as having moved fluid. Verify every protocol on labware and fluid you can afford to lose, keep a
hand on the power switch the first time the carrier moves, and please report what you find on the
[PyLabRobot forum](https://discuss.pylabrobot.org) so the warning can be removed.
```

```{warning}
Follow the manufacturer's installation, fluid-handling and safety instructions. Priming, purging and
dispensing all move fluid: the bottles must be full and the waste bottle empty enough before
anything in this notebook runs.
```

```{device-card} biotek-multiflo
```

## How it communicates

PyLabRobot frames each command as an 11-byte header and a payload, writes it to the instrument,
reads back an acknowledgement and the reply, and turns a non-zero status into a typed exception.
Which of the two transports carries those bytes is decided by the port string alone, and nothing
above that point knows which one it got.

Operations that move fluid do not answer when they are done. The driver sends them, then polls the
instrument's run state until it stops reporting a step in progress, which is why every method that
touches the instrument is awaited and can take as long as the physical operation does.

Reading `.LHC` protocol files additionally needs `pycryptodome`, which is not a PyLabRobot
dependency: the file format is encrypted, and the cipher is not in the standard library.

In [ ]:
%pip install "pylabrobot[serial]" pycryptodome

# On USB, install the FTDI dependencies instead: "pylabrobot[ftdi]".

## Physical setup and finding the port

Install, plumb and power the dispenser according to the manufacturer's instructions. Fit the tubing
cassettes into the pumps you intend to use, connect each syringe to the bottle it draws from,
connect the waste bottle, and connect the instrument to the computer.

Then find the port string:

- **Serial.** Pass the operating system's own name for the port: `COM3` on Windows,
  `/dev/ttyUSB0` or `/dev/ttyS4` on Linux and macOS. Anything that is not a USB serial number is
  taken to be a serial port and passed through unexamined.

- **USB.** List the attached FTDI devices and use the reported serial number:

  ```bash
  python -m pylibftdi.examples.list_devices
  ```

  The port is then `ftdi:<serial>`. The form the instrument's own protocol files record,
  `USB MultiFlo sn:<serial>`, is accepted as well.

Keep the carrier and the area around it clear from here on.

## Turn on logging

The driver reports what it is doing through the standard library's `logging`, and says nothing
otherwise. Without this cell the untested-driver warning below, and every step the instrument runs,
pass silently.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## See which ports have something behind them

`pyserial` lists every serial port the operating system offers, most of which are kernel
placeholders with no hardware behind them. Skipping those leaves the ports worth trying, and a
USB-serial adapter names the instrument it is wired to.

In [ ]:
from serial.tools.list_ports import comports

candidates = [port for port in comports() if port.description != "n/a" or port.vid is not None]

for port in candidates:
    serial_number = f"  sn:{port.serial_number}" if port.serial_number else ""
    print(f"{port.device:16} {port.description}{serial_number}")

if not candidates:
    print("no port has a device behind it; is the dispenser powered and connected?")

## Build the dispenser

Constructing the object opens nothing and touches no hardware. It records which port to use, which
model this is, and what to call the instrument in logs and error messages.

In [ ]:
from pylabrobot.agilent.biotek.lhc import MultiFlo

# The port is a serial one unless it carries a device serial number: on USB, pass
# "ftdi:YOUR_SERIAL" instead.
device = MultiFlo(port="COM3", name="MultiFlo")
device

## Connect

`setup()` opens the link, asks whether anything is listening, and reads the options the instrument
has fitted. That read is not optional: every step is encoded against it and every check measures
against it, so a failure here stops the notebook rather than being carried past.

It raises a `BiotekError` if the port will not open, if nothing answers on it, or if the fitted
options cannot be read.

In [ ]:
await device.setup()

## Ask what answered

**The model is declared, not discovered.** The instrument does not report which model it is, so it
is the class you constructed that decides how every step is encoded and which options are read.
`device.settings.family` echoes what was declared, not what is attached.

What the instrument does report is its serial number, and a firmware version record whose part
number says which instrument the installed firmware is built for. A MultiFlo reports a part number
beginning `721`, and `setup()` has already refused a link whose firmware says otherwise.

In [ ]:
print("serial number:  ", await device.get_serial_number())

version = await device.get_firmware_version()
print("part number:    ", version.part_number)
print("firmware:       ", version.software_version)
print("data version:   ", version.data_version)

## Read the configuration

What `setup()` read is kept as a read-only record of what somebody fitted. It is read-only because
the instrument is: of its whole command vocabulary almost nothing about the configuration can be
written, so a record that could be edited would only mislead.

On this model the record is short, because there is less to fit: the syringe box and its manifold,
and whether each peristaltic pump is there. The wash manifold, valve box and the modules that go
with them are only asked about on the models that can carry them, so they read as not fitted here
without a query being sent.

In [ ]:
settings = device.settings

print("family:              ", settings.family.name)
print("syringe box:         ", settings.syringe_box.name)
print("syringe box size:    ", settings.syringe_box_size.name)
print("syringe manifold:    ", settings.syringe_manifold.name)
print("primary peri pump:   ", settings.peri_pump)
print("secondary peri pump: ", settings.peri_pump_2)
print("half-µL volumes:     ", settings.half_ul_enabled)
print("wash manifold:       ", settings.washer_manifold.name)

## Ask what it can run

A model can be built to run a fixed set of operations; a particular instrument runs the subset its
fitted hardware supports. `get_available_steps()` has already applied that, which makes it the
answer to "why was my step refused" before you have written the step.

This model's whole palette is six step types: both syringe steps, all three peristaltic ones, and
shake/soak. There are no manifold steps and no strip washer steps to be had, whatever is fitted.

In [ ]:
for step_type in device.get_available_steps():
    print(step_type.name)

### What the oldest firmware does not check

A step type this firmware never knew is refused outright. What is worth knowing is the other
direction: eleven of the rules the newer models apply are **absent** here, because the features they
guard did not exist yet — single-well dispensing, the strip washer, the small-plate syringe
manifolds and the mini-tube carrier.

Most of those are unreachable anyway on a model that offers neither strip steps nor random access.
One is not: an ordinary peristaltic dispense reaches the single-well rule on newer firmware and is
measured against it, and here it is not. So a step this instrument accepts is not proof the same
step would be accepted by a MultiFlo FX.

## Tell it which plate is on the carrier

Nothing runs until a plate has been set. Every step carries the height it works at, measured from
the nominal heights of the format on the carrier, so without one there is nothing to measure from
and the driver raises `RejectedError` rather than guessing.

The format is resolved from the PyLabRobot plate resource itself — its columns, its rows, and how
deep its wells are. This model works nine formats, from 96- and 384-well plates through their
deep-well variants to 1536-well ones. The half-well, mini-tube, PCR and flanged formats share their
shape with an ordinary plate and differ in something a resource does not carry, so they are only
ever named outright.

In [ ]:
from pylabrobot.resources import cor_96_wellplate_360uL_Fb

plate = cor_96_wellplate_360uL_Fb(name="plate")
device.set_plate(plate)

print(device.plate)

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.plates.plate_type import PlateType

device.set_plate(plate, plate_type=PlateType.PLATE_96_HALF_WELL)
print(device.plate)

device.set_plate(plate)  # back to what it resolves to on its own

## Check before running

`can_run()` measures steps against the instrument as it is now — what is fitted, which plate is on
the carrier, and what the plate will accept — and touches nothing. It is what `run_protocol()` does
first, so calling it yourself is how you see a refusal without moving anything.

The report is truthy when everything can run, and prints as the list of what cannot. A volume the
syringe will not meter is refused with the range it would accept; that floor moves with the plate on
the carrier and the flow rate, so read it off the message rather than memorising it.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.peri_prime import PeriPrime
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.syringe_dispense import SyringeDispense

print(await device.can_run([PeriPrime(volume=300, peri_pump="Primary")]))
print(await device.can_run([SyringeDispense(volume=1, syringe="A")]))

## Prime and dispense from the syringes

Priming draws fluid through a syringe until its lines are full. It is the safest operation to try
first — nothing is dispensed into the plate — but it does move fluid, so check the bottles first.

A prime drives **one** syringe: unlike a dispense it cannot run both at once. Each syringe draws
from one of its two bottles, named per step, and stays committed to that bottle for the whole run
unless the syringe-side switching valve is fitted. Whether a dispense may drive both at once is
decided by the fitted manifold; on the plain 16-tube one each step drives a single syringe.

**The dispense fills the plate**, so put one on the carrier you are willing to fill.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.instrument.syringe_box_type import SyringeBoxType

if device.settings.syringe_box is not SyringeBoxType.NOT_INSTALLED:
    async with device.batch(home_on_close=True):
        await device.syringe_dispenser.prime(volume=5_000, syringe="A", syringe_bottle="A1")
        await device.syringe_dispenser.dispense(volume=50, syringe="A", syringe_bottle="A1")
else:
    print("no syringe box fitted; nothing to do")

## Prime, purge and dispense from the peristaltic pumps

A peristaltic pump pushes fluid through a tubing cassette. Priming fills the tubing; purging empties
it, running fluid to waste rather than into the plate — which is what you want at the end of a run
and before a cassette comes out. Both take either a volume per tube or a duration: give `duration`
and the step runs for that many seconds instead of metering a volume.

A step names the pump it wants and may name the cassette it needs. `"Any"` accepts whatever is
fitted; naming `"1uL"`, `"5uL"` or `"10uL"` is a requirement checked before anything moves, and two
steps wanting different cassettes in the same pump are refused on the second of the pair.

In [ ]:
async with device.batch(home_on_close=True):
    await device.peristaltic_dispenser.prime(volume=300, peri_pump="Primary")
    await device.peristaltic_dispenser.dispense(volume=100, peri_pump="Primary")
    if device.settings.peri_pump_2:
        await device.peristaltic_dispenser.dispense(
            volume=100, flow_rate="Low", cassette_type="5uL", peri_pump="Secondary"
        )

await device.peristaltic_dispenser.purge(volume=300, peri_pump="Primary")

## Shake and soak

Shaking and soaking are one step, and either half can be left out by giving it no time: `duration`
shakes, `soak_duration` leaves the plate still afterwards. Both are in seconds. This is the one
operation every instrument in the family offers, whatever is fitted.

In [ ]:
await device.shake(duration=30, intensity="Medium", axis="X", soak_duration=30)

## Working part of a plate

Every step that reaches into the plate takes a `columns` selection, and most take a `rows` one as
well. Both default to everything.

A selection is a `WellMask`: one entry per position, `1` to work it and `0` to skip it, and
`from_columns` builds one from column numbers counted from one. Rows go in sections of eight rather
than one at a time — a plate has `rows // 8` of them — and they bite only where the head covers less
than the plate, which for this model means the peristaltic cassettes on a plate of 384 wells or
more. Selecting no column is allowed and dispenses nothing; an empty row selection is refused where
the step checks rows.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.step_parts.masks import WellMask

first_two = WellMask.from_columns([1, 2])
print("selected columns:", first_two.selected_count)

await device.peristaltic_dispenser.dispense(volume=100, peri_pump="Primary", columns=first_two)

## Where in the well a step works

Every operation that reaches into the plate takes a `positioning`, and defaults it to the nominal
position for that head over the format on the carrier. Three numbers:

- `z_steps` — how deep the dispense tubes go. **This is a height, not an offset**: it defaults to
  the plate record's own nominal height for the head, and a larger number reaches further down into
  the well. On this model it must be 1-1500.
- `x_steps` — across the well, ±125.
- `y_steps` — along the well, ±40.

The nominal heights come from the plate record, so they change with the plate.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.step_parts.positioning import Positioning

print("nominal dispensing height:", device.plate.dispenser_height)

await device.peristaltic_dispenser.dispense(
    volume=100,
    peri_pump="Primary",
    positioning=Positioning(
        z_steps=device.plate.dispenser_height - 20,  # 20 steps higher in the well
        x_steps=15,  # toward one side
        y_steps=0,
    ),
)

```{note}
The offsets are in motor steps rather than millimetres, which is the one place this package deviates
from PyLabRobot's convention — the field names say so. The conversion differs per axis, per model
and per head, and only part of it is established, so the package does not convert. A step is also
what a protocol file stores, which is what lets a protocol be read and written with no instrument to
ask.
```

## Run several operations in one batch

Opening a batch homes the motors, takes the instrument so that nothing else can interleave a run on
it, and holds it until the block ends. Every operation opens one; doing it once around several
operations — as the cells above have been doing — is what stops the motors being homed between each
of them.

`home_on_close=True` drives the transport home before the batch closes. The instrument does not do
this by itself; ask for it when the next thing to touch the plate is a person. Nesting is allowed
and does nothing: an operation called inside an open batch joins it.

In [ ]:
async with device.batch(home_on_close=True):
    await device.peristaltic_dispenser.prime(volume=300, peri_pump="Primary")
    await device.peristaltic_dispenser.dispense(volume=100, peri_pump="Primary")
    await device.shake(duration=15, soak_duration=0)

## Watch a step, and stop it

`get_status()` reports what the instrument is doing, which timed phase a running step is in, and how
many seconds are left in it. It can be called at any time, including while a step is running.

An operation does not return until its step has finished, so pausing or aborting means asking from
somewhere else while it runs. In a notebook that is a task. Abort makes the waiting operation raise
`AbortedError`, so a protocol run ends where it was stopped rather than carrying on.

In [ ]:
import asyncio

from pylabrobot.agilent.biotek.lhc.error_handling import AbortedError

running = asyncio.create_task(
    device.peristaltic_dispenser.prime(duration=30, peri_pump="Primary")
)
await asyncio.sleep(2)

status = await device.get_status()
print("state:    ", status.state.name)
print("activity: ", status.activity.name)
print("remaining:", status.remaining, "s")

await device.abort()
try:
    await running
except AbortedError as error:
    print("stopped:", error)

## Run a protocol file

A `.LHC` protocol file is read into a `Protocol`: what it will run, and everything the file records
alongside it. Reading needs `pycryptodome`, installed at the top of this notebook.

Reading a file never fails on a step it cannot understand — the file's own records are kept as they
are, so a protocol from another model still reads, prints and writes. `build_steps()` is what turns
those records into steps, and it names the one that will not read. A file written for a newer model
may well contain a step this instrument cannot run; `can_run()` is what says so.

In [ ]:
from pylabrobot.agilent.biotek.lhc import read

protocol = read("path/to/your/protocol.LHC")

print("name:      ", protocol.protocol_name)
print("written by:", protocol.lhc_version)
print("written for:", protocol.instrument_name)
print("plate:     ", protocol.plate_type or protocol.plate_type_number)
print(
    "entries:   ",
    len(protocol.entries),
    "of which",
    len(protocol.device_entries),
    "operate the instrument",
)

for index, step in enumerate(protocol.build_steps()):
    print(f"  step {index}: {type(step).__name__}")

### What the file says about its instrument

A protocol file records the options the instrument had fitted when it was written. Nothing runs
against that record — steps are encoded against the instrument in front of you — and nothing writes
it to the instrument. It is good for exactly one question, worth asking about a file that came from
another machine: was this written for a differently equipped dispenser? A file written for a
two-pump instrument will not run on a one-pump one.

`compare_settings()` is truthy when the two agree, and prints as the options that differ. It raises
`ValueError` for a file that carries no such record, which is how the oldest releases wrote one.

In [ ]:
comparison = device.compare_settings(protocol)
print(comparison)
print("same configuration:", bool(comparison))

### Running it

`run_protocol()` checks the protocol, opens one batch around the whole run, sends each step and
polls it to completion. The check is the same `can_run()` from above and happens automatically, so a
protocol that cannot run raises before anything moves.

**This runs whatever the protocol does**, which for most dispenser protocols means filling every
well of the plate on the carrier. Read the steps printed above first.

The entries that sequence a run rather than operate the instrument — delays, loops, remarks — are
not run; the device steps go in file order. They are still on `protocol.entries` to inspect.

In [ ]:
await device.run_protocol(protocol, home_on_close=True)

Steps built in Python run the same way. `run_protocol()` takes a list of steps as readily
as a protocol, and `run_step()` runs a single one.

In [ ]:
await device.run_protocol(
    [
        PeriPrime(volume=300, peri_pump="Primary"),
        SyringeDispense(volume=50, syringe="A", syringe_bottle="A1"),
    ],
    home_on_close=True,
)

## Home the transport and disconnect

Homing drives the transport to its home position and confirms it arrived. Do it before a person
reaches for the plate, unless the last batch already closed with `home_on_close=True`.

`stop()` closes the link. It does nothing on an instrument that is already closed, so it is safe to
run this cell twice, and it is worth running from a `finally` in a script so that a failed run does
not leave the port open.

In [ ]:
await device.home()
await device.stop()

```{note}
The fluid left in the syringes and the cassettes after a run is the instrument's problem, not the
driver's. Follow the manufacturer's shutdown and maintenance procedure — purging each pump
(`device.peristaltic_dispenser.purge(...)`) is what empties a cassette before it comes out, and most
maintenance routines ship as protocol files you can run with `run_protocol()`.
```